In [1]:
print("Hello World!")

Hello World!


In [2]:
# TODO: spojiti data.txt i labels.txt u jedan data.csv (prepakovani podaci) i proveriti strukturu
# ocekujem da bude 4480 redova, 40 razlicitih subjekata, pa za svakog po 112 redova merenja, i za svakog subjekta po 28 merenja iz svake aktivnost
# odnosno, klase su balansirane
# ocekujem 535 kolona, od kojih su 533 atributi koji ucestvuju u klasifikaciji, prvi atribut je id subjekta koji ucestvuje u podeli i uzorkovanju podataka,
# poslednji atribut je ciljna klasa (1-neutral, 2-emontional, 3-mental, 4-physical)
# nema nedostajucih vrednosti, svi atributi su realne vrednosti, i ciljna klasa koja je kategorickog tipa je vec kodirana
# atributi za EDA signale iz ruke i sake imaju ista imena (jesu isti, ali za razlicite delove tela), pa treba dodati prefikse kako bi se znalo o kojem tacno signalu je rec
# prvo idu 104 EDA signala vezanih za ruku, pa potom 104 EDA signala vezana za saku

csv_filename = "data/data.csv"

# granice blokova atributa (indeksi kolona, 0-bazirani):
# 0 - subject_id, 1..174 - ECG, 175..325 - TEB, 326..429 - EDA ruka, 430..533 - EDA saka, 534 - target
EDA_ARM_START = 326
EDA_HAND_START = 430

def transform_to_csv(labels: str = "", data: str = ""):
    with open(labels, "r") as f_labels:
        with open(data, "r") as f_data:
            labels = f_labels.readlines()
            labels = list(map(str.strip, labels))
            data = f_data.readlines()
            data = list(map(str.strip, data))
            labels = labels[:-1]
            # zamenjujemo naziv poslednjeg ciljnog atributa u 'target'
            labels.append("target")
            # zamenjujemo naziv prvog atributa u subject_id
            labels[0] = "subject_id"

            # EDA podaci za ruku i saku imaju ista imena, pa ih transformisemo dodavanjem dela tela na kojem su mereni
            # opsezi se ne smeju preklapati, inace bi kolona na granici dobila oba prefiksa (npr. EDA_Hand_Arm_...)
            for i in range(EDA_ARM_START, EDA_HAND_START):
                prefix, sufix = labels[i].split("_", maxsplit=1)
                labels[i] = prefix + "_Arm_" + sufix
            for i in range(EDA_HAND_START, len(labels) - 1):
                prefix, sufix = labels[i].split("_", maxsplit=1)
                labels[i] = prefix + "_Hand_" + sufix

            print(labels[:EDA_ARM_START])
            print(labels[EDA_ARM_START:EDA_HAND_START])
            print(labels[EDA_HAND_START:-1])
            csv_data = data
            csv_data.insert(0, ",".join(labels))
            with open(csv_filename, "w") as f_csv:
                f_csv.writelines(f"{item}\n" for item in csv_data)

transform_to_csv(labels="data/labels.txt", data="data/data.txt")

['subject_id', 'ECG_original_mean', 'ECG_original_std', 'ECG_original_trimmean25', 'ECG_original_median', 'ECG_original_skewness', 'ECG_original_kurtosis', 'ECG_original_max', 'ECG_original_min', 'ECG_original_prctile25', 'ECG_original_prctile75', 'ECG_original_geomean(abs)', 'ECG_original_harmmean', 'ECG_original_mad', 'ECG_original_baseline', 'ECG_RR_window_mean', 'ECG_RR_window_std', 'ECG_RR_window_trimmean25', 'ECG_RR_window_median', 'ECG_RR_window_skewness', 'ECG_RR_window_kurtosis', 'ECG_RR_window_max', 'ECG_RR_window_min', 'ECG_RR_window_prctile25', 'ECG_RR_window_prctile75', 'ECG_RR_window_geomean(abs)', 'ECG_RR_window_harmmean', 'ECG_RR_window_mad', 'ECG_RR_window_baseline', 'ECG_amplitude_RR_mean', 'ECG_amplitude_RR_std', 'ECG_amplitude_RR_trimmean25', 'ECG_amplitude_RR_median', 'ECG_amplitude_RR_skewness', 'ECG_amplitude_RR_kurtosis', 'ECG_amplitude_RR_max', 'ECG_amplitude_RR_min', 'ECG_amplitude_RR_prctile25', 'ECG_amplitude_RR_prctile75', 'ECG_amplitude_RR_geomean(abs)', '

In [26]:
import pandas as pd
import numpy as np

# provera da ispravno pravimo data.csv fajl od polaznih fajlova koji predstavljaju dataset kako bi olaksali upotrebu i manipulaciju dataset-om
df = pd.read_csv(csv_filename)
print(df.head())

print("Provera broja NaN/null vrednosti:")
print(df.isna().sum().sum())

print("Provera broja inf vrednosti za numericke atribute:")
print(np.isinf(df.select_dtypes(include=np.number)).sum().sum())

print("Provera broja redova i kolona: ")
print(df.shape)

print("Provera broja subjekata i balansiranosti podataka po subjektu: ")
print(df[["subject_id", "target"]].value_counts())

print("Provera balansiranosti klasa: ")
print(df["target"].value_counts())

print("Provera broja atributa po vrsti merenja:")
print(f"ECG: {df.columns.str.startswith("ECG").sum()}")
print(f"TIB: {df.columns.str.startswith("IT").sum()}")
print(f"EDA arm: {df.columns.str.startswith("EDA_Arm").sum()}")
print(f"EDA hand: {df.columns.str.startswith("EDA_Hand").sum()}")

   subject_id  ECG_original_mean  ECG_original_std  ECG_original_trimmean25  \
0           1          -0.004125          0.254095                 0.001426   
1           1           0.031029          0.193761                 0.012918   
2           1           0.015678          0.182336                -0.003028   
3           1           0.014525          0.176636                -0.006161   
4           1           0.010349          0.179248                -0.008526   

   ECG_original_median  ECG_original_skewness  ECG_original_kurtosis  \
0             -0.01037              -0.538509                5.95534   
1             -0.00237               0.781415                5.18794   
2             -0.02337               0.881194                5.66530   
3             -0.02737               1.024900                6.10968   
4             -0.02737               0.935697                5.83902   

   ECG_original_max  ECG_original_min  ECG_original_prctile25  ...  \
0           1.04063   